# False Positive Benchmark Analysis

Local VS Code notebook for the real-only deepfake false-positive benchmark.

This notebook reads the centralized benchmark CSV and optional threshold sweep summary CSVs from local paths only. It does not run model inference and does not edit the main benchmark CSV.

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path(r"/Users/subrat/Desktop/Deepfake")
OUTPUT_BASE_DIR = PROJECT_ROOT / "output"
MODEL_FILE_PREFIX = "cf_vit"


def latest_legacy_csv(base_dir: Path = OUTPUT_BASE_DIR) -> Path:
    patterns = ["benchmark_*.csv", "false_positive_complete_benchmark_*.csv"]
    matches = []
    for pattern in patterns:
        matches.extend(base_dir.glob(pattern))
    matches = [path for path in set(matches) if path.name != "benchmark_cf_vit.csv"]
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No benchmark CSV found under {base_dir}")
    return matches[0]


MAIN_CSV_PATH = OUTPUT_BASE_DIR / "benchmark_cf_vit.csv"
if not MAIN_CSV_PATH.exists():
    MAIN_CSV_PATH = latest_legacy_csv()

THRESHOLD_SWEEP_DIR = OUTPUT_BASE_DIR / "sweep"
legacy_sweep_matches = sorted(OUTPUT_BASE_DIR.glob("sweep_cf_vit*"), key=lambda p: p.stat().st_mtime, reverse=True)
LEGACY_THRESHOLD_SWEEP_DIR = legacy_sweep_matches[0] if legacy_sweep_matches else OUTPUT_BASE_DIR / "sweep_cf_vit"
OUTPUT_DIR = OUTPUT_BASE_DIR / "analysis"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_THRESHOLD = 0.50
THRESHOLDS = [round(x / 100, 2) for x in range(10, 100, 5)]

print(f"MAIN_CSV_PATH: {MAIN_CSV_PATH}")
print(f"THRESHOLD_SWEEP_DIR: {THRESHOLD_SWEEP_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


## Setup

This cell installs only the plotting and data packages needed for local analysis if they are missing. It does not install or load the deepfake model.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "kaleido": "kaleido",
}

missing = [package for import_name, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required analysis packages are already installed.")

## Load Benchmark Data

This loads the centralized real-only benchmark CSV and prepares analysis columns from saved probability scores. Real-only labels mean `FP` is a real image predicted as fake, and `TN` is a real image predicted as real.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

if not MAIN_CSV_PATH.exists():
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV_PATH}")

df = pd.read_csv(MAIN_CSV_PATH, low_memory=False)

if "source_subgroup" not in df.columns:
    df["source_subgroup"] = ""
df["raw_source"] = df.get("source", "").fillna("").astype(str)
df["source_subgroup"] = df["source_subgroup"].fillna("").astype(str)
source_subgroup_first = df["source_subgroup"].str.replace("\\", "/", regex=False).str.split("/").str[0]
source_wrapper_mask = df["raw_source"].eq("data") & source_subgroup_first.ne("")
df["source"] = df["raw_source"]
df.loc[source_wrapper_mask, "source"] = source_subgroup_first[source_wrapper_mask]


for col in ["test_type", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level", "error"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)

numeric_cols = [
    "normal_same_resolution_fake_probability",
    "normal_original_fake_probability",
    "stress_fake_probability",
    "score_delta_vs_clean",
    "score_delta_vs_original",
    "variant_width",
    "variant_height",
    "variant_megapixels",
    "original_megapixels",
    "brightness_mean",
    "contrast_std",
    "blur_score",
    "sharpness_score",
    "file_size_mb",
    "inference_ms",
]
for col in numeric_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid = df[df["error"].eq("")].copy()
normal = valid[(valid["test_type"] == "normal") & valid["normal_same_resolution_fake_probability"].notna()].copy()
stress = valid[(valid["test_type"] == "stress") & valid["stress_fake_probability"].notna()].copy()

normal["score"] = normal["normal_same_resolution_fake_probability"]
stress["score"] = stress["stress_fake_probability"]
scored = pd.concat([normal.assign(scope="normal"), stress.assign(scope="stress")], ignore_index=True, sort=False)
scored["prediction_at_default"] = np.where(scored["score"] >= DEFAULT_THRESHOLD, "FP", "TN")

print(f"Loaded rows: {len(df):,}")
print(f"Completed normal score rows: {len(normal):,}")
print(f"Completed stress score rows: {len(stress):,}")
print(f"Default threshold: {DEFAULT_THRESHOLD}")

## Threshold Sweep Summaries

If summary CSVs already exist, this notebook loads them. If not, it computes threshold summaries from saved probability scores only. It does not run model inference and does not add rows to the centralized CSV.

In [ ]:
SWEEP_SOURCE_DIR = THRESHOLD_SWEEP_DIR if THRESHOLD_SWEEP_DIR.exists() else LEGACY_THRESHOLD_SWEEP_DIR
SWEEP_FILES = {
    "overall": SWEEP_SOURCE_DIR / "threshold_sweep_overall.csv",
    "by_source": SWEEP_SOURCE_DIR / "threshold_sweep_by_source.csv",
    "by_resolution": SWEEP_SOURCE_DIR / "threshold_sweep_by_resolution.csv",
    "by_source_resolution": SWEEP_SOURCE_DIR / "threshold_sweep_by_source_resolution.csv",
    "by_stress_type_level": SWEEP_SOURCE_DIR / "threshold_sweep_by_stress_type_level.csv",
}

SUMMARY_COLUMNS = [
    "threshold", "test_scope", "group_name", "group_value", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level",
    "count", "FP", "TN", "FPR", "TNR", "score_mean", "score_median", "score_p95", "score_min", "score_max",
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP", "TN_to_TN_rate", "TN_to_FP_rate", "FP_to_TN_rate", "FP_to_FP_rate",
    "delta_vs_clean_mean", "delta_vs_clean_median", "delta_vs_clean_p95", "delta_vs_clean_min", "delta_vs_clean_max",
]

SWEEP_NUMERIC_COLUMNS = [
    "threshold", "count", "FP", "TN", "FPR", "TNR", "score_mean", "score_median", "score_p95", "score_min", "score_max",
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP", "TN_to_TN_rate", "TN_to_FP_rate", "FP_to_TN_rate", "FP_to_FP_rate",
    "delta_vs_clean_mean", "delta_vs_clean_median", "delta_vs_clean_p95", "delta_vs_clean_min", "delta_vs_clean_max",
]


def normalize_sweep_frame(frame):
    frame = frame.copy()
    for col in SWEEP_NUMERIC_COLUMNS:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    for col in ["test_scope", "group_name", "group_value", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level"]:
        if col in frame.columns:
            frame[col] = frame[col].fillna("").astype(str)
    return frame

def _single_value(frame, col):
    vals = [str(v) for v in frame[col].dropna().unique().tolist() if str(v) != ""] if col in frame.columns else []
    return vals[0] if len(vals) == 1 else ""

def _metric_row(frame, threshold, scope, group_name, group_value):
    scores = pd.to_numeric(frame.get("score", pd.Series(dtype=float)), errors="coerce").dropna()
    count = int(len(scores))
    fp = int((scores >= threshold).sum()) if count else 0
    tn = count - fp
    row = {
        "threshold": threshold, "test_scope": scope, "group_name": group_name, "group_value": group_value,
        "source": _single_value(frame, "source"), "resolution_bucket": _single_value(frame, "resolution_bucket"),
        "target_dimension": _single_value(frame, "target_dimension"), "resize_mode": _single_value(frame, "resize_mode"),
        "stress_type": _single_value(frame, "stress_type"), "stress_level": _single_value(frame, "stress_level"),
        "count": count, "FP": fp, "TN": tn, "FPR": fp / count if count else np.nan, "TNR": tn / count if count else np.nan,
        "score_mean": scores.mean() if count else np.nan, "score_median": scores.median() if count else np.nan,
        "score_p95": scores.quantile(0.95) if count else np.nan, "score_min": scores.min() if count else np.nan, "score_max": scores.max() if count else np.nan,
        "TN_to_TN": np.nan, "TN_to_FP": np.nan, "FP_to_TN": np.nan, "FP_to_FP": np.nan,
        "TN_to_TN_rate": np.nan, "TN_to_FP_rate": np.nan, "FP_to_TN_rate": np.nan, "FP_to_FP_rate": np.nan,
        "delta_vs_clean_mean": np.nan, "delta_vs_clean_median": np.nan, "delta_vs_clean_p95": np.nan,
        "delta_vs_clean_min": np.nan, "delta_vs_clean_max": np.nan,
    }
    if scope == "stress" and count:
        pairs = pd.DataFrame({
            "clean": pd.to_numeric(frame["normal_same_resolution_fake_probability"], errors="coerce"),
            "stress": pd.to_numeric(frame["stress_fake_probability"], errors="coerce"),
        }).dropna()
        n = int(len(pairs))
        if n:
            clean_fp = pairs["clean"] >= threshold
            stress_fp = pairs["stress"] >= threshold
            transitions = {
                "TN_to_TN": int((~clean_fp & ~stress_fp).sum()),
                "TN_to_FP": int((~clean_fp & stress_fp).sum()),
                "FP_to_TN": int((clean_fp & ~stress_fp).sum()),
                "FP_to_FP": int((clean_fp & stress_fp).sum()),
            }
            for key, value in transitions.items():
                row[key] = value
                row[f"{key}_rate"] = value / n
        deltas = pd.to_numeric(frame["score_delta_vs_clean"], errors="coerce").dropna()
        if not deltas.empty:
            row["delta_vs_clean_mean"] = deltas.mean()
            row["delta_vs_clean_median"] = deltas.median()
            row["delta_vs_clean_p95"] = deltas.quantile(0.95)
            row["delta_vs_clean_min"] = deltas.min()
            row["delta_vs_clean_max"] = deltas.max()
    return row

def _build_summary(frame, scope, group_name, group_cols):
    rows = []
    for threshold in THRESHOLDS:
        if frame.empty:
            continue
        if not group_cols:
            rows.append(_metric_row(frame, threshold, scope, group_name, "all"))
        else:
            for keys, group in frame.groupby(group_cols, dropna=False):
                if not isinstance(keys, tuple):
                    keys = (keys,)
                row = _metric_row(group, threshold, scope, group_name, "|".join(str(k) for k in keys))
                for col, value in zip(group_cols, keys):
                    if col in row:
                        row[col] = value
                rows.append(row)
    return pd.DataFrame(rows, columns=SUMMARY_COLUMNS)

def compute_sweep_summaries():
    THRESHOLD_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    global SWEEP_SOURCE_DIR, SWEEP_FILES
    SWEEP_SOURCE_DIR = THRESHOLD_SWEEP_DIR
    SWEEP_FILES = {key: THRESHOLD_SWEEP_DIR / path.name for key, path in SWEEP_FILES.items()}
    all_scored = pd.concat([normal.assign(score=normal["score"], scope="normal"), stress.assign(score=stress["score"], scope="stress")], ignore_index=True, sort=False)
    summaries = {
        "overall": pd.concat([_build_summary(normal, "normal", "overall", []), _build_summary(stress, "stress", "overall", [])], ignore_index=True),
        "by_source": pd.concat([_build_summary(normal, "normal", "source", ["source"]), _build_summary(stress, "stress", "source", ["source"])], ignore_index=True),
        "by_resolution": pd.concat([_build_summary(normal, "normal", "resolution", ["resolution_bucket"]), _build_summary(stress, "stress", "resolution", ["resolution_bucket"])], ignore_index=True),
        "by_source_resolution": pd.concat([_build_summary(normal, "normal", "source_resolution", ["source", "resolution_bucket"]), _build_summary(stress, "stress", "source_resolution", ["source", "resolution_bucket"])], ignore_index=True),
        "by_stress_type_level": _build_summary(stress, "stress", "stress_type_level", ["stress_type", "stress_level"]),
    }
    for key, frame in summaries.items():
        frame.to_csv(SWEEP_FILES[key], index=False)
    return summaries

sweep = {key: normalize_sweep_frame(frame) for key, frame in compute_sweep_summaries().items()}
print("Refreshed threshold sweep summaries from saved probability scores.")

for key, frame in sweep.items():
    print(f"{key}: {frame.shape}")

## Plot Helpers

These helpers make the notebook presentation-ready: stable category order, explicit `No Samples` handling, fixed rate scales, and compact labels. The charts use only saved probability scores from the CSV.

In [ ]:
SOURCE_ORDER = sorted([src for src in df["source"].dropna().astype(str).unique().tolist() if src])
if not SOURCE_ORDER:
    SOURCE_ORDER = ["unknown"]

NORMAL_PROCESSING_ORDER = [
    "clean original",
    "clean 1024 aspect", "clean 1024 square",
    "clean 720 aspect", "clean 720 square",
    "clean 512 aspect", "clean 512 square",
    "clean 256 aspect", "clean 256 square",
]

STRESS_TYPE_ORDER = ["blur", "brightness", "sharpness", "contrast", "jpeg_compression"]
STRESS_LEVEL_ORDER = {
    "blur": ["low", "medium", "high"],
    "brightness": ["darker", "brighter"],
    "sharpness": ["lower", "higher"],
    "contrast": ["lower", "higher"],
    "jpeg_compression": ["q95", "q80", "q60", "q40"],
}
STRESS_VARIANT_ORDER = [f"{stress_type} {level}" for stress_type in STRESS_TYPE_ORDER for level in STRESS_LEVEL_ORDER[stress_type]]
PROCESSING_VARIANT_ORDER = NORMAL_PROCESSING_ORDER + STRESS_VARIANT_ORDER
TRANSITION_ORDER = ["TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP"]
TRANSITION_COLORS = {
    "TN_to_TN": "#4C78A8",
    "TN_to_FP": "#E45756",
    "FP_to_TN": "#72B7B2",
    "FP_to_FP": "#F58518",
}
MODEL_COLORS = {"clean": "#4C78A8", "stress": "#E45756"}


def save_current_fig(name):
    path = OUTPUT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


def no_data_chart(name, message):
    plt.figure(figsize=(10, 3.5))
    plt.text(0.5, 0.5, message, ha="center", va="center", fontsize=13, color="#374151")
    plt.axis("off")
    save_current_fig(name)


def threshold_label(threshold=DEFAULT_THRESHOLD):
    return f"Threshold {threshold:.2f}"


def add_processing_variant(frame):
    frame = frame.copy()
    target = frame["target_dimension"].fillna("").astype(str)
    mode = frame["resize_mode"].fillna("").astype(str)
    clean_label = np.where(target.eq("original"), "clean original", "clean " + target + " " + mode)
    stress_label = (frame["stress_type"].fillna("").astype(str) + " " + frame["stress_level"].fillna("").astype(str)).str.strip()
    frame["processing_variant"] = np.where(frame["scope"].eq("stress"), stress_label, clean_label)
    frame["processing_variant"] = frame["processing_variant"].replace("", "unknown")
    return frame


def source_processing_metrics(frame, threshold=DEFAULT_THRESHOLD):
    rows = []
    if not frame.empty:
        frame = add_processing_variant(frame)
        grouped = frame.groupby(["source", "processing_variant"], dropna=False)
    else:
        grouped = None

    for source in SOURCE_ORDER:
        for variant in PROCESSING_VARIANT_ORDER:
            if grouped is not None and (source, variant) in grouped.groups:
                group = grouped.get_group((source, variant))
                scores = pd.to_numeric(group["score"], errors="coerce").dropna()
            else:
                scores = pd.Series(dtype=float)
            count = int(len(scores))
            fp = int((scores >= threshold).sum()) if count else 0
            tn = count - fp
            rows.append({
                "source": source,
                "processing_variant": variant,
                "count": count,
                "FP": fp,
                "TN": tn,
                "FPR": fp / count if count else np.nan,
                "label": "No Samples" if count == 0 else f"{fp}/{count}",
            })
    return pd.DataFrame(rows)


def source_stress_type_metrics(threshold=DEFAULT_THRESHOLD):
    rows = []
    valid_stress = stress.dropna(subset=["stress_fake_probability"]).copy() if not stress.empty else pd.DataFrame()
    grouped = valid_stress.groupby(["source", "stress_type"], dropna=False) if not valid_stress.empty else None
    for source in SOURCE_ORDER:
        for stress_type in STRESS_TYPE_ORDER:
            if grouped is not None and (source, stress_type) in grouped.groups:
                group = grouped.get_group((source, stress_type))
                scores = pd.to_numeric(group["stress_fake_probability"], errors="coerce").dropna()
            else:
                scores = pd.Series(dtype=float)
            count = int(len(scores))
            fp = int((scores >= threshold).sum()) if count else 0
            rows.append({
                "source": source,
                "stress_type": stress_type,
                "count": count,
                "FP": fp,
                "FPR": fp / count if count else np.nan,
                "label": "No Samples" if count == 0 else f"{fp}/{count}",
            })
    return pd.DataFrame(rows)


def prepare_heatmap_values(metrics, index_col, column_col, value_col="FPR", label_col="label"):
    value = metrics.pivot(index=index_col, columns=column_col, values=value_col)
    labels = metrics.pivot(index=index_col, columns=column_col, values=label_col)
    value = value.astype(float)
    return value, labels

## Chart 1: Source × Processing Variant FPR Heatmap

**What it shows:** False-positive rate for every source and expected processing variant, including clean resize variants and stress variants.

**Why it is useful:** It exposes fragile source/processing combinations while clearly showing missing combinations as `No Samples`.

**Insight to look for:** Warm cells with nonzero `FP/count`, especially in stress variants or smaller clean resize modes.

**How to interpret results:** Colored numeric cells have samples. Gray `No Samples` cells are absent combinations and should not be treated as good performance.

In [ ]:
if scored.empty:
    no_data_chart("01_source_processing_fpr_heatmap", "No completed score rows available.")
else:
    metrics = source_processing_metrics(scored, DEFAULT_THRESHOLD)
    value, labels = prepare_heatmap_values(metrics, "source", "processing_variant")
    value = value.reindex(index=SOURCE_ORDER, columns=PROCESSING_VARIANT_ORDER)
    labels = labels.reindex(index=SOURCE_ORDER, columns=PROCESSING_VARIANT_ORDER).fillna("No Samples")
    mask = value.isna()
    plot_values = value.fillna(0.0)

    plt.figure(figsize=(15, 1.2 + 0.65 * len(SOURCE_ORDER)))
    ax = sns.heatmap(
        plot_values,
        mask=mask,
        cmap="rocket_r",
        vmin=0,
        vmax=1,
        linewidths=0.4,
        linecolor="white",
        cbar_kws={"label": "FPR"},
    )
    ax.set_facecolor("#f3f4f6")
    ax.set_title(f"Source × Processing Variant FPR ({threshold_label()})", pad=12, weight="bold")
    ax.set_xlabel("Processing variant")
    ax.set_ylabel("Source")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    # Sparse, high-contrast annotations: missing cells and nonzero false-positive cells only.
    text_box = dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="none", alpha=0.78)
    for y, source in enumerate(value.index):
        for x, variant in enumerate(value.columns):
            row = metrics[(metrics["source"].eq(source)) & (metrics["processing_variant"].eq(variant))]
            if row.empty:
                continue
            count = int(row.iloc[0]["count"])
            fp = int(row.iloc[0]["FP"])
            if count == 0:
                ax.text(x + 0.5, y + 0.5, "No\nSamples", ha="center", va="center", fontsize=6, color="#111827", bbox=text_box)
            elif fp > 0:
                ax.text(x + 0.5, y + 0.5, f"{fp}/{count}", ha="center", va="center", fontsize=7, color="#111827", bbox=text_box)
    save_current_fig("01_source_processing_fpr_heatmap")

## Chart 2: Source-Wise Fake-Probability Distribution

**What it shows:** Fake-probability distribution for each source, with median/spread/outliers and a visible mean marker.

**Why it is useful:** It shows whether a source has higher typical scores, high-score tails, or a mean close to the threshold.

**Insight to look for:** Sources whose box, outliers, or mean diamond sit near the red threshold line.

**How to interpret results:** Scores above the red line are false positives. The black diamond is the mean fake-probability score for each source.

In [ ]:
if scored.empty:
    no_data_chart("02_source_score_boxplot", "No completed score rows available.")
else:
    plot_df = scored.copy()
    plot_df["source"] = pd.Categorical(plot_df["source"], categories=SOURCE_ORDER, ordered=True)
    score_series = pd.to_numeric(plot_df["score"], errors="coerce")
    plot_df["score"] = score_series
    stats = plot_df.groupby("source", observed=False).agg(
        count=("score", "count"),
        mean_score=("score", "mean"),
    ).reindex(SOURCE_ORDER)

    plt.figure(figsize=(10.5, 5.2))
    ax = sns.boxplot(
        data=plot_df,
        x="source",
        y="score",
        order=SOURCE_ORDER,
        color="#A6CEE3",
        width=0.56,
        showfliers=True,
        flierprops={"marker": "o", "markersize": 3, "markerfacecolor": "#4b5563", "markeredgecolor": "#4b5563", "alpha": 0.45},
    )
    ax.axhline(DEFAULT_THRESHOLD, color="#D62728", linestyle="--", linewidth=1.4, label=threshold_label())
    ax.set_ylim(0, 1)

    for i, source in enumerate(SOURCE_ORDER):
        count = 0 if pd.isna(stats.loc[source, "count"]) else int(stats.loc[source, "count"])
        mean_score = stats.loc[source, "mean_score"]
        if count == 0 or pd.isna(mean_score):
            ax.text(i, 0.5, "No\nSamples", ha="center", va="center", fontsize=9, color="#6b7280")
            continue
        ax.scatter(i, mean_score, s=80, marker="D", color="#111827", edgecolor="white", linewidth=0.8, zorder=5, label="Mean" if i == 0 else None)
        if count > 0:
            ax.text(i, min(0.97, mean_score + 0.055), f"μ={mean_score:.3f}", ha="center", va="bottom", fontsize=8, color="#111827")
        ax.text(i, 0.03, f"n={count}", ha="center", va="bottom", fontsize=8, color="#374151")

    ax.set_title("Source-Wise Fake-Probability Distribution", pad=12, weight="bold")
    ax.set_xlabel("Source")
    ax.set_ylabel("Fake probability")
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc="upper right", frameon=True)
    plt.xticks(rotation=25, ha="right")
    save_current_fig("02_source_score_boxplot")

## Chart 3: Overall And Source-Wise Threshold Sweep

**What it shows:** Overall false-positive rate plus per-source false-positive rate from threshold 0.10 to 0.95.

**Why it is useful:** The overall curve gives a simple operating view; source curves show whether one category remains fragile even when the average looks safe.

**Insight to look for:** Sources above the overall curve or lines that remain high at stricter thresholds.

**How to interpret results:** Lower curves are safer. The vertical red line marks the current default threshold.

In [ ]:
if scored.empty:
    no_data_chart("03_threshold_sweep", "No completed score rows available.")
else:
    rows = []
    all_scores = pd.to_numeric(scored["score"], errors="coerce").dropna()
    for threshold in THRESHOLDS:
        all_count = int(len(all_scores))
        all_fp = int((all_scores >= threshold).sum()) if all_count else 0
        rows.append({"threshold": threshold, "source": "Overall", "count": all_count, "FPR": all_fp / all_count if all_count else np.nan})
        for source in SOURCE_ORDER:
            source_scores = pd.to_numeric(scored.loc[scored["source"].eq(source), "score"], errors="coerce").dropna()
            count = int(len(source_scores))
            fp = int((source_scores >= threshold).sum()) if count else 0
            rows.append({"threshold": threshold, "source": source, "count": count, "FPR": fp / count if count else np.nan})
    sweep_source = pd.DataFrame(rows)

    plt.figure(figsize=(10.8, 5.2))
    ax = plt.gca()
    source_part = sweep_source[~sweep_source["source"].eq("Overall")]
    sns.lineplot(data=source_part, x="threshold", y="FPR", hue="source", hue_order=SOURCE_ORDER, marker="o", linewidth=1.8, alpha=0.9, ax=ax)
    overall_part = sweep_source[sweep_source["source"].eq("Overall")]
    ax.plot(overall_part["threshold"], overall_part["FPR"], color="#111827", linewidth=3.0, marker="o", label="Overall")
    ax.axvline(DEFAULT_THRESHOLD, color="#D62728", linestyle="--", linewidth=1.2)
    ax.set_title("Overall and Source-Wise FPR Threshold Sweep", pad=12, weight="bold")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("False positive rate")
    ax.set_ylim(0, 1)
    handles, labels = ax.get_legend_handles_labels()
    # Put Overall first.
    if "Overall" in labels:
        order = [labels.index("Overall")] + [i for i, label in enumerate(labels) if label != "Overall"]
        handles = [handles[i] for i in order]
        labels = [labels[i] for i in order]
    ax.legend(handles, labels, title="Source", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    save_current_fig("03_threshold_sweep")

## Chart 4: Clean vs Stress Line Chart By Source

**What it shows:** Mean clean baseline score and mean stressed score for each stress variant, with one subplot per source.

**Why it is useful:** It reveals whether a stress type pushes real images upward compared with the matched clean baseline.

**Insight to look for:** Red stress lines above blue clean lines, especially with large gaps.

**How to interpret results:** A higher stress line means the transformation increases fake probability. `No Samples` means stress inference for that source/variant is not available yet.

In [ ]:
valid_stress = stress.dropna(subset=["normal_same_resolution_fake_probability", "stress_fake_probability"]).copy() if not stress.empty else pd.DataFrame()
if valid_stress.empty:
    no_data_chart("04_clean_vs_stress_by_source", "No completed stress rows with matched clean baseline scores available.")
else:
    valid_stress["stress_variant"] = (valid_stress["stress_type"].astype(str) + " " + valid_stress["stress_level"].astype(str)).str.strip()
    rows = []
    grouped = valid_stress.groupby(["source", "stress_variant"], dropna=False)
    for source in SOURCE_ORDER:
        for variant in STRESS_VARIANT_ORDER:
            if (source, variant) in grouped.groups:
                group = grouped.get_group((source, variant))
                rows.append({
                    "source": source,
                    "stress_variant": variant,
                    "clean_mean": pd.to_numeric(group["normal_same_resolution_fake_probability"], errors="coerce").mean(),
                    "stress_mean": pd.to_numeric(group["stress_fake_probability"], errors="coerce").mean(),
                    "count": len(group),
                })
            else:
                rows.append({"source": source, "stress_variant": variant, "clean_mean": np.nan, "stress_mean": np.nan, "count": 0})
    line_df = pd.DataFrame(rows)
    n_sources = len(SOURCE_ORDER)
    fig, axes = plt.subplots(n_sources, 1, figsize=(13, max(3, 2.25 * n_sources)), sharex=True, sharey=True)
    if n_sources == 1:
        axes = [axes]
    x = np.arange(len(STRESS_VARIANT_ORDER))
    for ax, source in zip(axes, SOURCE_ORDER):
        part = line_df[line_df["source"].eq(source)].set_index("stress_variant").reindex(STRESS_VARIANT_ORDER)
        if part["count"].fillna(0).sum() == 0:
            ax.text(0.5, 0.5, "No Samples", transform=ax.transAxes, ha="center", va="center", color="#6b7280")
        ax.plot(x, part["clean_mean"], marker="o", linewidth=1.8, color=MODEL_COLORS["clean"], label="Clean baseline")
        ax.plot(x, part["stress_mean"], marker="o", linewidth=1.8, color=MODEL_COLORS["stress"], label="Stress")
        ax.axhline(DEFAULT_THRESHOLD, color="#D62728", linestyle="--", linewidth=1.0)
        ax.set_title(source, loc="left", fontsize=11, weight="bold")
        ax.set_ylim(0, 1)
        ax.set_ylabel("Mean score")
        ax.grid(True, axis="y", alpha=0.25)
    axes[0].legend(loc="upper right", frameon=True)
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(STRESS_VARIANT_ORDER, rotation=35, ha="right", fontsize=8)
    fig.suptitle("Clean Baseline vs Stress Mean Fake Probability by Source", y=1.01, weight="bold")
    save_current_fig("04_clean_vs_stress_by_source")

## Chart 5: Source × Stress Type FPR Heatmap

**What it shows:** False-positive rate by source and stress type, aggregated across stress levels and dimensions.

**Why it is useful:** It identifies which stress families are most fragile for each source.

**Insight to look for:** Warm cells in a source row or stress-type column.

**How to interpret results:** Gray `No Samples` cells have no completed stress scores. Numeric cells show false positives among available stress rows.

In [ ]:
metrics = source_stress_type_metrics(DEFAULT_THRESHOLD)
value, labels = prepare_heatmap_values(metrics, "source", "stress_type")
value = value.reindex(index=SOURCE_ORDER, columns=STRESS_TYPE_ORDER)
labels = labels.reindex(index=SOURCE_ORDER, columns=STRESS_TYPE_ORDER).fillna("No Samples")
mask = value.isna()
if metrics["count"].sum() == 0:
    no_data_chart("05_source_stress_type_fpr_heatmap", "No completed stress score rows available.")
else:
    plt.figure(figsize=(9, 1.2 + 0.65 * len(SOURCE_ORDER)))
    ax = sns.heatmap(
        value.fillna(0.0),
        mask=mask,
        cmap="rocket_r",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "FPR"},
    )
    ax.set_facecolor("#f3f4f6")
    ax.set_title(f"Source × Stress Type FPR ({threshold_label()})", pad=12, weight="bold")
    ax.set_xlabel("Stress type")
    ax.set_ylabel("Source")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
    text_box = dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="none", alpha=0.78)
    for y, source in enumerate(value.index):
        for x, stress_type in enumerate(value.columns):
            row = metrics[(metrics["source"].eq(source)) & (metrics["stress_type"].eq(stress_type))]
            if row.empty:
                continue
            count = int(row.iloc[0]["count"])
            fp = int(row.iloc[0]["FP"])
            if count == 0:
                ax.text(x + 0.5, y + 0.5, "No\nSamples", ha="center", va="center", fontsize=7, color="#111827", bbox=text_box)
            elif fp > 0:
                ax.text(x + 0.5, y + 0.5, f"{fp}/{count}", ha="center", va="center", fontsize=8, color="#111827", bbox=text_box)
    save_current_fig("05_source_stress_type_fpr_heatmap")

## Chart 6: Stress Transition Percentages By Source

**What it shows:** Per-source percentage split across `TN_to_TN`, `TN_to_FP`, `FP_to_TN`, and `FP_to_FP` transitions.

**Why it is useful:** Percentages make sources comparable even when they have different stress sample counts.

**Insight to look for:** Large red `TN_to_FP` shares, which mean clean real images became false positives after stress.

**How to interpret results:** Each source bar sums to 100% when samples exist. Sources without completed stress rows are marked `No Samples`.

In [ ]:
transition_counts = pd.DataFrame(0, index=SOURCE_ORDER, columns=TRANSITION_ORDER, dtype=int)
valid_stress = stress.dropna(subset=["normal_same_resolution_fake_probability", "stress_fake_probability"]).copy() if not stress.empty else pd.DataFrame()
if not valid_stress.empty:
    clean_fp = valid_stress["normal_same_resolution_fake_probability"] >= DEFAULT_THRESHOLD
    stress_fp = valid_stress["stress_fake_probability"] >= DEFAULT_THRESHOLD
    valid_stress["transition_at_default"] = np.select(
        [~clean_fp & ~stress_fp, ~clean_fp & stress_fp, clean_fp & ~stress_fp, clean_fp & stress_fp],
        TRANSITION_ORDER,
        default="unknown",
    )
    counted = valid_stress.groupby(["source", "transition_at_default"]).size().unstack(fill_value=0)
    for col in TRANSITION_ORDER:
        if col not in counted.columns:
            counted[col] = 0
    transition_counts = counted.reindex(SOURCE_ORDER, fill_value=0)[TRANSITION_ORDER].astype(int)

if valid_stress.empty:
    no_data_chart("06_stress_transition_percentages", "No completed stress rows with matched clean baseline scores available.")
else:
    totals = transition_counts.sum(axis=1)
    transition_percent = transition_counts.div(totals.replace(0, np.nan), axis=0).fillna(0.0) * 100
    ax = transition_percent.plot(
        kind="bar",
        stacked=True,
        figsize=(10.8, 5.2),
        color=[TRANSITION_COLORS[c] for c in TRANSITION_ORDER],
        edgecolor="white",
        linewidth=0.45,
    )
    ax.set_title(f"Stress Prediction Transitions by Source ({threshold_label()})", pad=12, weight="bold")
    ax.set_xlabel("Source")
    ax.set_ylabel("Share of stress samples")
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
    ax.legend(title="Transition", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    plt.xticks(rotation=25, ha="right")

    for i, source in enumerate(SOURCE_ORDER):
        total = int(totals.loc[source]) if source in totals.index else 0
        if total == 0:
            ax.text(i, 50, "No\nSamples", ha="center", va="center", fontsize=9, color="#111827")
            continue
        bottom = 0.0
        for transition_name in TRANSITION_ORDER:
            pct = float(transition_percent.loc[source, transition_name])
            if pct >= 8:
                ax.text(i, bottom + pct / 2, f"{pct:.0f}%", ha="center", va="center", fontsize=8, color="white")
            bottom += pct
        ax.text(i, 101, f"n={total}", ha="center", va="bottom", fontsize=8, color="#374151")
    save_current_fig("06_stress_transition_percentages")